# Семинар 1. 

## Работа с API PyMC6

%pip install -U pymc

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm
import pytensor.tensor as pt

In [ ]:
RANDOM_SEED = 8927
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-darkgrid")

## 1. Создание моделей

Модели в PyMC задаются с помощью класса `Model`. Внутри класса задаются все случайные величины (RVs) и расчитываются логарифмы заданных случайных величин и их градиенты. Обычно модель задают через ключевое слово `with`:

In [ ]:
with pm.Model() as model:
    # Model definition
    pass

Далее мы поговорим про RVs более подробно, а пока давайте создадим простую модель, чтобы изучить класс `Model`.

In [ ]:
with pm.Model() as model:
    mu = pm.Normal("mu", mu=0, sigma=1)
    obs = pm.Normal("obs", mu=mu, sigma=1, observed=rng.standard_normal(100))

In [ ]:
model.basic_RVs

$$p(\Theta)$$

In [ ]:
model.free_RVs

$$
p(D|\Theta)
$$

In [ ]:
model.observed_RVs

In [ ]:
fn  = model.compile_logp()

In [ ]:
model.initial_point()

In [ ]:
fn(model.initial_point())

In [ ]:
model.debug()

In [ ]:
# задаем вот таким образом фиксированный набор наблюдений (лучше писать в дальнейшем так)
obs = rng.standard_normal(100)

with pm.Model() as model_invalid:
    mu = pm.LogNormal("mu", mu=0, sigma=1)
    obs = pm.Normal("obs1234123", mu=0, sigma=mu, observed=obs)

In [ ]:
model_invalid.debug()

Важно объяснить, почему мы решили применить в моделировании `logp`.  Выше можно увидеть, что `logp` вызывают с аргументами, таким образом задавая распределение в модели. Точнее, она возвращает функцию от текущего состояния, или состояния, заданного в качестве аргумента для `logp` (как в примере ниже).

По разным причинам, определение `Model` не статично. Если необходимо применить `logp` внутри цикла и нужно задать его статично, можно использовать конструкцию вида `logp = model.logp`. Ниже пример, как это сделать $-$ отметье, как ускоряется работа и работает кэширование:

In [ ]:
%timeit model.compile_logp()({"mu": 0.1})
logp = model.compile_logp()
%timeit logp({"mu": 0.1})

## 2. Вероятностные распределения

Каждая вероятностная модель состоит из наблюдаемых и ненаблюдаемых случайных величин (RVs). Наблюдаемые случайные величины задаются через правдоподобие, а ненаблюдаемы случайные величины определяются априорными распределениями. В PyMC, структура вероятностных распределений выглядит следующим образом:

[`pymc:api_distributions`](https://www.pymc.io/projects/examples/en/latest/object_index/distributions.html)
- [`pymc:api_distributions_continuous`](https://www.pymc.io/projects/examples/en/latest/object_index/distributions/continuous.html)
- [`pymc:api_distributions_discrete`](https://www.pymc.io/projects/examples/en/latest/object_index/distributions/continuous.html)
- [`pymc:api_distributions_multivariate`](https://www.pymc.io/projects/examples/en/latest/object_index/distributions/continuous.html)
- [`pymc:api_distributions_mixture`](https://www.pymc.io/projects/docs/en/latest/api/distributions/mixture.html)
- [`pymc:api_distributions_timeseries`](https://www.pymc.io/projects/docs/en/latest/api/distributions/timeseries.html)
- [`pymc:api_distributions_censored`](https://www.pymc.io/projects/docs/en/latest/api/distributions/censored.html)
- [`pymc:api_distributions_simulator`](https://www.pymc.io/projects/docs/en/latest/api/distributions/simulator.html)

### Ненаблюдаемые случайные величины

Каждая ненаблюдаемая случайная величина задается следующим образом: `name (str)`, а затем основные параметры распределения. Таким образом, нормальное априорное распределение может быть задано следующим образом:

In [ ]:
with pm.Model() as model:
    x = pm.Normal("x", mu=0, sigma=1)
    y = pm.Normal("y", mu=x, sigma=0.1)
    # x2 = x**2 + y
    x2 = x*y

In [ ]:
samples = pm.draw([x2, x, y], draws=100)

In [ ]:
plt.scatter(samples[0], samples[1]);

Для модели можно вычислить логарифм вероятности:

In [ ]:
y

In [ ]:
pm.logp(x, 1).eval()

In [ ]:
# смотрим значения логарифмов плотностей для одной точки в пространстве параметров
model.point_logps()

### Наблюдаемые случайные величины

Наблюдаемые RVs задаются аналогично ненаблюдаемым RVs, но требуют, чтобы были фактические данные, которые передаются в аргумент `observed`:

In [ ]:
# акцент на том, что можно использовать observed в модели, а можно использовать pm.Data и менять данные на лету
observed = rng.standard_normal(100)


with pm.Model() as model:
    obs = pm.Normal("x", mu=0, sigma=1, observed=observed)

In [ ]:
model.free_RVs

In [ ]:
with model:
    pm.sample()

`observed` поддерживает структуры  `list`, `numpy.ndarray` и `pytensor`.

In [ ]:
with pm.Model() as model:
    mu = pm.Normal("mu")
    obs = pm.Normal("x", mu=mu, sigma=1, observed=rng.standard_normal(100))

In [ ]:
model.free_RVs

In [ ]:
with model:
    trace = pm.sample()

In [ ]:
trace

### Алгебраические преобразования

PyMC позволяет применять алгебраические преобразования с RVs разными способами:

In [ ]:
with pm.Model():
    x = pm.Normal("x", mu=0, sigma=1)
    y = pm.Gamma("y", alpha=1, beta=1)
    plus_2 = x + 2
    summed = x + y
    squared = x**2
    sined = pm.math.sin(x)

Хотя эти преобразования проводятся без каких либо проблем, их результаты не сохраняются автоматически. Поэтому, если нужно отслеживать значения таких преобразований, нужно применить `pm.Deterministic`:

In [ ]:
with pm.Model():
    x = pm.Normal("x", mu=0, sigma=1)
    # plus_2 = x + 2  
    plus_2 = pm.Deterministic("x plus 2", x + 2) # тут, по сути, происходит логирование, без этого цепочка просто не сохраняется в trace
    trace = pm.sample()

In [ ]:
trace

Заметьте, что `plus_2` можно применять и в функции выше, нужно только указать PyMC, что нужно сохранять значения этой RV для нас.

### Лист RV / многомерные RV

В примерах выше мы создавала скалярные RV. Однако иногда в моделях мы хотим иметь несколько RV. Иногда пользователи библиотеки могут создавать листы RV, например так:

In [ ]:
with pm.Model() as model:
    # плохо:
    x = [pm.Normal(f"x_{i}", mu=0, sigma=1) for i in range(10)]

In [ ]:
pm.model_to_graphviz(model)

Такой способ рабочий, но медленный, поэтому мы его не рекомендуем. Вместо него можно использовать [`coordinates <pymc:dimensionality>`](https://www.pymc.io/projects/docs/en/latest/learn/core_notebooks/dimensionality.html):

In [ ]:
coords = {"cities": ["Santiago", "Mumbai", "Tokyo"]}
with pm.Model(coords=coords) as model:
    # хорошо:
    x = pm.Normal("x", mu=0, sigma=1, dims="cities")

In [ ]:
pm.model_to_graphviz(model)

`x` теперь $-$ массив длинной 3, и каждый из 3 элементов массива имеет собственный лейбл.  Это сильно упрощает идентификацию 3 различных переменных, когда мы изучаем результаты моделирования. К элементам массива можно обращаться по индексу и проводить с ними алгебраические операции:

In [ ]:
with model:
    y = x[0] * x[1]  # поддержка индексации
    x.dot(x.T)  # поддержка линейной алгебры

In [ ]:
import pytensor.tensor as pt

## 3. Вывод модели

После того, как мы определили нашу модель, мы должны вывести её для получения апостериорных распределний. Основной класс выводов PyMC $-$ сэмплирование.

### 3.1 Сэмплирование

Сэмплирование методом Монте-Карло (MCMC)  производится функцией `pm.sample()`. По умолчанию, функция пытается подобрать сэмплер автоматически.  `pm.sample()` возвращает объект класса `arviz.InferenceData`.  Объекты `InferenceData`  легко экспортируются и импортируются и могут содержать дополнительную информацию (в том числе и мета-информацию: дата сэмплирования, версия пакета и тд). Для дополнительного изучения ознакомьтесь с [`ArviZ Quickstart <arviz:quickstart>`](https://python.arviz.org/en/stable/user_guide/getting_started.html).

In [ ]:
with pm.Model() as model:
    mu = pm.Normal("mu", mu=0, sigma=1)
    obs = pm.Normal("obs", mu=mu, sigma=1, observed=rng.standard_normal(100))

    idata = pm.sample(2000)

Как видите, если модель содержит только непрерывные переменные, PyMC назначает сэмплер NUTS, который очень эффективен даже для сложных моделей. PyMC также выполняет начальную настройку (tuning), чтобы подобрать хорошие стартовые параметры для сэмплера. Здесь мы получаем 2000 сэмплов из апостериорного распределения для каждой цепи и позволяем сэмплеру настраивать свои параметры в течение дополнительных 1500 итераций.

По умолчанию строится 4 цепи Маркова, однако их число можно регулировать через параметр `chains`

In [ ]:
idata.observed_data

По умолчанию сэмплы, полученные во время настройки (tuning), отбрасываются. При `discard_tuned_samples=False` их можно сохранить — они попадут в отдельную группу внутри объекта `InferenceData` (то есть в `idata.warmup_posterior`). 

Управлять параллельным запуском цепей можно через аргументы `chains` и `cores`:

In [ ]:
with pm.Model() as model:
    sigma = pm.LogNormal("mu")
    obs = pm.Normal("obs", sigma = 1) * sigma  

    idata = pm.sample()

In [ ]:
az.plot_dist(idata);

In [ ]:
az.plot_trace_dist(idata);

In [ ]:
az.plot_pair(idata, 
             visuals={"divergence": True}, 
             aes_by_visuals={"divergence": {"color": "red"}}
             );

In [ ]:
idata.posterior["mu"]

In [ ]:
# Получаем значения одной цепи
idata.posterior["mu"].sel(chain=2)#.shape

### 3.2 Анализ результатов сэмплирования

Наиболее часто используемый график для анализа результатов сэмплирования — так называемый trace-plot:

In [ ]:
with pm.Model(coords=dict(trial=range(5))) as model:
    mu = pm.Normal("mu", mu=[-1, 2, 4, 2, 5], sigma=1, dims="trial")

    idata = pm.sample()

In [ ]:
az.plot_trace_dist(idata);

Другая часто используемая метрика — статистика Гельмана-Рубина, или R-hat

In [ ]:
# Хорошо, когда ess_bulk/ess_tail > 200 и r_hat близок к 1. Если другие значения, то это повод задуматься
az.summary(idata)

Наконец, для построения графика апостериорного распределения в стиле, вдохновлённом работой [`kruschke2014doing`](https://www.uvm.edu/~bbeckage/Teaching/PBIO_294/Kruschke-DoingBayesian2ndEd-Figures.pdf), можно воспользоваться следующим:

In [ ]:
az.plot_dist(idata);

Для многомерных моделей становится неудобно просматривать трейсы всех параметров. При использовании NUTS можно взглянуть на energy plot, чтобы оценить проблемы со сходимостью

In [ ]:
with pm.Model() as model:
    mu = pm.Normal("mu", mu=0, sigma=1)
    observed_date = pm.Normal("obs", mu=mu, sigma=1, observed=rng.standard_normal(100))
    prior = pm.sample_prior_predictive()
    az.plot_ppc_dist(prior, group="prior_predictive", visuals={"observed_dist": True});

In [ ]:
with pm.Model() as model:
    mu = pm.Normal("mu", mu=0, sigma=1)
    obs = pm.Normal("obs", mu=mu, sigma=1, observed=100+rng.standard_normal(100))
    prior = pm.sample_prior_predictive()
    az.plot_ppc_dist(prior, group="prior_predictive", visuals={"observed_dist": True});

In [ ]:
with pm.Model() as model:
    mu = pm.Normal("mu", mu=0, sigma=1)
    obs = pm.Normal("obs", mu=mu, sigma=1, observed=100*rng.standard_normal(100))
    prior = pm.sample_prior_predictive()
    az.plot_ppc_dist(prior, group="prior_predictive", visuals={"observed_dist": True});

In [ ]:
with pm.Model() as model:
    mu = pm.Normal("mu", mu=0, sigma=1)
    sigma = pm.LogNormal("sigma", np.log(1.4), sigma=1)
    obs = pm.Normal("obs", mu=mu, sigma=sigma, observed=1/10*rng.standard_normal(100))
    prior = pm.sample_prior_predictive()
    az.plot_ppc_dist(prior, group="prior_predictive", visuals={"observed_dist": True});

In [ ]:
with model:
    trace = pm.sample()
    pm.sample_posterior_predictive(trace, extend_inferencedata=True) # тут переписал, так как на PYMC 6.0 уже не работает

In [ ]:
az.plot_dist(trace)

In [ ]:
az.plot_ppc_dist(trace);

In [ ]:
with pm.Model() as model:
    mu = pm.Normal("mu", mu=0, sigma=100)
    
    obs = pm.Normal("obs", mu=mu, sigma=1, observed=100+rng.standard_normal(100))
    prior = pm.sample_prior_predictive()
    az.plot_ppc_dist(prior, group="prior_predictive", visuals={"observed_dist": True});

In [ ]:
with pm.Model() as model:
    mu = pm.Normal("mu", mu=0, sigma=100)
    sigma = pm.LogNormal("sigma", np.log(1.4), sigma=0.1)
    obs = pm.Normal("obs", mu=mu, sigma=sigma, observed=100+rng.standard_normal(100))
    prior = pm.sample_prior_predictive()
    az.plot_ppc_dist(prior, group="prior_predictive", visuals={"observed_dist": True});

In [ ]:
with pm.Model(coords={"idx": np.arange(100)}) as model:
    x = pm.Normal("x", mu=0, sigma=1, dims="idx")
    idata = pm.sample()

az.plot_energy(idata, show_bfmi=False);

## 4. Posterior Predictive Sampling

Функция `sample_posterior_predictive()` выполняет предсказание на отложенных (hold-out) данных и постериорные предиктивные проверки (posterior predictive checks)

In [ ]:
data = rng.standard_normal(100)
with pm.Model() as model:
    mu = pm.Normal("mu", mu=0, sigma=1)
    sd = pm.HalfNormal("sd", sigma=1)
    obs = pm.Normal("obs", mu=mu, sigma=sd, observed=data)

    idata = pm.sample()

In [ ]:
with model:
    pm.sample_posterior_predictive(idata, extend_inferencedata=True)

In [ ]:
pc = az.plot_ppc_dist(idata)
ax = pc.viz["plot"].ds.to_array().values.item()
ax.axvline(data.mean(), ls="--", color="r", label="True mean")
ax.legend(fontsize=10)

## 4.1 Предсказание на отложенных данных

Во многих случаях требуется делать предсказания на отложенных (hold-out) данных. Это особенно актуально в вероятностном машинном обучении (Probabilistic Machine Learning) и байесовском глубоком обучении (Bayesian Deep Learning). PyMC предоставляет контейнер `pm.MutableData`, который помогает в таких случаях. Это обёртка вокруг переменной `pytensor.shared`, позволяющая позже изменять значения данных. В остальном объекты `pm.MutableData` можно использовать точно так же, как любой другой numpy-массив или тензор.


Это различие существенно, поскольку внутри все модели в PyMC представляют собой гигантские символьные выражения. Когда вы передаёте в модель необработанные данные напрямую, вы даёте PyTensor разрешение считать эти данные константой и, если это имеет смысл, оптимизировать их (то есть исключить из вычислений). Если позже вам понадобится изменить эти данные, у вас может не оказаться способа обратиться к ним внутри большого символьного выражения. Использование `pm.MutableData` даёт возможность указать на конкретное место в символьном выражении и изменить то, что там находится.



In [ ]:
x = rng.standard_normal(100)
y = x > 0

coords = {"idx": np.arange(100)}
with pm.Model() as model:
    # create shared variables that can be changed later on
    x_obs = pm.Data("x_obs", x, dims="idx")
    y_obs = pm.Data("y_obs", y, dims="idx")

    coeff = pm.Normal("x", mu=0, sigma=1)
    logistic = pm.math.sigmoid(coeff * x_obs)
    pm.Bernoulli("obs", p=logistic, observed=y_obs, dims="idx")
    idata = pm.sample()

Теперь предположим, что мы хотим сделать предсказание на ранее не использованных данных. Для этого нам надо заменить значения `x_obs` и `y_obs`. Теоретически, нам не нужно задавать `y_obs`, так как мы предполагаем предсказывать его; но размерность должна соответствовать `x_obs`.

In [ ]:
with model:
    # изменяем значения и размерности данных
    pm.set_data(
        {
            "x_obs": [-1, 0, 1.0],
            # используем фиктивные переменные соответсвующей размерности:
            "y_obs": [False, False, False],
        },
        coords={"idx": [1001, 1002, 1003]},
    )

    pm.sample_posterior_predictive(idata, extend_inferencedata=True)

In [ ]:
idata.posterior_predictive["obs"].mean(dim=["draw", "chain"])